# Analyse der Datensatzgröße & Datenbereinigung

## 1. Ursachen für die Reduzierung der Datenmenge auf 13.676 Einträge

Die Reduzierung des ursprünglichen "MovieLens 25M"-Datensatzes (der ca. 62.000 Filme umfasst) auf exakt **13.676 Filme** im bereinigten Datensatz resultiert aus notwendigen Filterungsschritten in unserer Daten-Pipeline:

- **Verfügbarkeit des Tag-Genomes**: Die für das Mood-Training essenziellen Stimmungs-Vektoren (das Tag-Genome) sind im MovieLens-Datensatz nur für knapp 14.000 Filme berechnet worden. Filme ohne diese Vektoren besitzen keine Labels und mussten über einen `inner join` entfernt werden.
- **Ausschluss fehlender Filmbeschreibungen**: Da das Modell lernen soll, aus Freitext Stimmungen vorherzusagen, ist eine Textbeschreibung zwingend erforderlich. Filme, für die aus TMDB keine Beschreibung (`overview`) geladen werden konnte, wurden herausgefiltert.

## 2. Eignung der verbleibenden Datenmenge für das Fine-Tuning

Obwohl 13.676 Datensätze für das Training eines neuronalen Netzes von Grund auf unzureichend wären, ist diese Menge für unsere Methode optimal:

- **Transfer Learning**: Wir nutzen ein vortrainiertes Transformer-Modell (z. B. DistilBERT), das bereits über umfassendes sprachliches Vorwissen verfügt.
- **Optimaler Bereich (Sweet Spot)**: Für das Fine-Tuning eines solchen Modells gilt eine dichte, qualitativ hochwertige Datenbasis von 10.000 bis 20.000 Beispielen als ideal. Sie bietet ausreichend Varianz für robustes Lernen bei gleichzeitig sehr kurzen Trainingszeiten und minimalem Rauschen.

## 2. Code-Erklärung & wichtige Code-Zitate

Um mit einem geeigneten Datensatz zu arbeiten, wurden heterogene Datensätze (MovieLens 25M und TMDB) aggregiert und in einem Datensatz zusammengefasst. Dieser Datensatz wird für das Training genutzt und auf Huggingsface abgelegt. Damit soll ein übermäßiges Aufblähen der Repository-Größe verhindert werden. 

Die Aufbereitung und der Upload werden durch zwei Skripte im Verzeichnis `src/data/` durchgeführt:
### A. Datenaufbereitung in [prepare_data.py](file:///Users/sebastianwolf/projects/pythonics/waki-movies/src/data/prepare_data.py)

#### 1. Filterung nach Beschreibungen
Um sicherzustellen, dass nur Filme mit Textbeschreibungen verarbeitet werden, bereinigen wir die TMDB-Daten:
```python
# Just keep movies with an overview (text)
tmdb_df = tmdb_df[["id", "title", "overview"]].dropna(subset=["overview"])
```
*Erklärung:* Wir behalten nur die Spalten `id`, `title` und `overview` und entfernen alle Zeilen, in denen `overview` (der Filmtext) leer (`NaN`) ist.

#### 2. Zusammenführung der IDs
Da MovieLens und TMDB unterschiedliche IDs nutzen, führen wir sie über eine Mapping-Tabelle (`links.csv`) zusammen:
```python
# Connect TMDB data with MovieLens IDs
movies_df = pd.merge(tmdb_df, links_df, on="tmdbId", how="inner")
```
*Erklärung:* Der `inner join` stellt sicher, dass wir nur Filme behalten, die sowohl in TMDB (mit Text) als auch in MovieLens (für die Stimmungsverknüpfung) existieren.

#### 3. Binarisierung der Stimmungs-Relevanz
Das Tag-Genome enthält kontinuierliche Relevanz-Scores für Stimmungen. Um ein Multi-Label-Klassifikationsproblem zu definieren, binarisieren wir diese Werte mit einem Schwellenwert von `0.5`:
```python
pivot_df = scores_named.pivot(index="movieId", columns="tag", values="relevance")
pivot_df = (pivot_df >= 0.5).astype(float)
```
*Erklärung:* Wenn die Relevanz eines Stimmungs-Tags für einen Film $\ge 0.5$ ist, setzen wir das Label auf `1.0` (Trifft zu), andernfalls auf `0.0` (Trifft nicht zu).

#### 4. Finaler Merge
Zuletzt verbinden wir die Filmdaten (Titel, Text) mit den binären Stimmungs-Vektoren:
```python
# Merge vectors with movie texts
final_df = pd.merge(movies_df, pivot_df, on="movieId", how="inner")
```
*Erklärung:* Auch hier sorgt der `inner join` dafür, dass nur Filme übrig bleiben, die sowohl über einen Stimmungs-Vektor (`pivot_df`) als auch über Metadaten (`movies_df`) verfügen. Das Ergebnis sind genau die **13.676 Filme**.

---

### B. Hugging Face Upload in [hugging_face.py](file:///Users/sebastianwolf/projects/pythonics/waki-movies/src/data/hugging_face.py)

Sobald die CSV-Datei lokal unter `src/data/datasets/hf_folder/prepared_movie-data.csv` generiert wurde, wird sie mit folgendem Code hochgeladen:

```python
login(token=Settings.HF_ACCESS_TOKEN)

try:
    upload_folder(folder_path=path, repo_id=repo_id, repo_type="dataset")
    print(f"Dataset successfully uploaded to Hugging Face Hub: {repo_id}")
```

#### Erklärung:
1. `login(token=...)`: Authentifiziert den Account mit dem in der `.env`-Datei konfigurierten `HF_ACCESS_TOKEN`.
2. `upload_folder(...)`: Lädt den gesamten Ordner (in dem sich unsere präparierte CSV-Datei befindet) auf den Hugging Face Hub unter der angegebenen Repository-ID hoch. Dadurch ist der Datensatz cloudbasiert erreichbar und kann im Trainings-Skript effizient gestreamt werden.
